# T1.6 / T1.7 — Análise dos resultados (E1, E2 e E4)

Consolida os artefatos gerados pelos experimentos:

```bash
cd backend
python -m experiments.e1_no_drift
python -m experiments.e2_drift_levels
python -m experiments.e4_retraining
```

Os resultados ficam em `backend/experiments/results/{e1,e2,e4}`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RESULTS = ROOT / "backend" / "experiments" / "results"

RESULTS

## 1. E1 — estabilidade sem drift

In [ ]:
e1 = json.loads((RESULTS / "e1" / "e1_results.json").read_text(encoding="utf-8"))
print("Baseline:", e1["model"]["version"])
pd.Series(e1["baseline_test_metrics"]).to_frame("teste")

e1_windows = pd.read_csv(RESULTS / "e1" / "e1_windows.csv")
e1_windows[["window", "rows", "fraud_rate", "precision", "recall", "f1", "auc_roc"]]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(e1_windows["window"], e1_windows["f1"], marker="o", label="F1")
ax.plot(e1_windows["window"], e1_windows["auc_roc"], marker="s", label="AUC-ROC")
ax.set_ylim(0, 1.05)
ax.set_title("E1 — estabilidade do baseline sem drift")
ax.set_ylabel("métrica")
ax.legend()
plt.show()

## 2. E2 — degradação por cenário

In [ ]:
e2 = json.loads((RESULTS / "e2" / "e2_results.json").read_text(encoding="utf-8"))
e2_summary = pd.read_csv(RESULTS / "e2" / "e2_summary.csv")
e2_summary[["scenario", "intensity", "precision", "recall", "f1", "auc_roc", "pr_auc", "max_psi"]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.barplot(data=e2_summary, x="scenario", y="f1", ax=axes[0], palette="Reds_r")
axes[0].set_title("F1 por cenário de drift")
axes[0].set_ylim(0, 1.05)

sns.barplot(data=e2_summary, x="scenario", y="max_psi", ax=axes[1], palette="Blues")
axes[1].axhline(0.25, color="#dc2626", linestyle="--", label="limiar crítico")
axes[1].set_title("PSI máximo por cenário")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
e2_windows = pd.read_csv(RESULTS / "e2" / "e2_windows.csv")
fig, ax = plt.subplots(figsize=(10, 4))
sns.lineplot(data=e2_windows, x="window", y="f1", hue="scenario", marker="o", ax=ax)
ax.set_title("E2 — F1 por janela e cenário")
ax.set_ylim(0, 1.05)
plt.show()

## 3. E4 — recuperação via retreinamento por alerta

In [ ]:
e4 = json.loads((RESULTS / "e4" / "e4_results.json").read_text(encoding="utf-8"))
e4_windows = pd.read_csv(RESULTS / "e4" / "e4_windows.csv")
e4_retrains = pd.read_csv(RESULTS / "e4" / "e4_retrains.csv")

print(json.dumps(e4["summary"], indent=2, ensure_ascii=False))
e4_retrains[["window", "status", "version", "training_windows"]]

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(e4_windows["window"], e4_windows["f1_no_retrain"], marker="o", label="sem retreino")
ax.plot(e4_windows["window"], e4_windows["f1_alert"], marker="s", label="retreino por alerta")

for _, row in e4_retrains.iterrows():
    if row.get("status") == "completed":
        ax.axvline(row["window"], color="#16a34a", alpha=0.25)

ax.set_ylim(0, 1.05)
ax.set_title("E4 — F1 com e sem retreinamento (linhas verdes = retreinos)")
ax.set_ylabel("F1")
ax.legend()
plt.show()

## 4. Consolidação dos entregáveis

- **D1** — dataset processado/simulado (`data/paysim_sample.csv` ou gerador
  sintético determinístico).
- **D2** — modelo baseline versionado (`backend/models/v1.1.0-xgb.joblib`).
- **D6** — resultados de E1, E2 e E4 (JSON + CSV + relatório em
  `backend/experiments/results/`).